# Self-Pruning Neural Network for CIFAR-10

**Problem:** Build a feed-forward neural network that learns to prune its own weights
during training using learnable *gate* parameters and an L1 sparsity regularisation loss.

**Parts covered:**
1. Custom `PrunableLinear` layer with gated weights
2. Sparsity regularisation loss (L1 on sigmoid gates)
3. Training & evaluation for three λ values (low / medium / high)
4. Results table + gate-distribution plot
5. Written analysis

## Step 1 – Import Dependencies

In [1]:
import os
import time
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

print("PyTorch version:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found, running on CPU")

PyTorch version: 2.10.0+cu128
GPU: Tesla T4


## Step 2 – Custom `PrunableLinear` Layer

This is the core building block.  
Each weight has a matching `gate_score`.  
A **sigmoid** squashes the gate score to (0, 1).  
If the gate becomes ≈ 0 the weight is effectively removed (pruned).

**Gradient flow:** both `self.weight` and `self.gate_scores` are `nn.Parameter`,
so PyTorch auto-diff tracks gradients through both during `.backward()`.

In [2]:
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super(PrunableLinear, self).__init__()

        self.in_features  = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))

        self.gate_scores = nn.Parameter(torch.empty(out_features, in_features))

        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter("bias", None)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

        nn.init.uniform_(self.gate_scores, -0.1, 0.1)

        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1.0 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)

    def get_gates(self):
        gates = torch.sigmoid(self.gate_scores)
        return gates

    def forward(self, x):
        gates = self.get_gates()

        pruned_weights = self.weight * gates

        output = F.linear(x, pruned_weights, self.bias)
        return output

    def extra_repr(self):
        return f"in={self.in_features}, out={self.out_features}, bias={self.bias is not None}"


temp_layer = PrunableLinear(8, 4)
temp_input = torch.randn(2, 8)
temp_out   = temp_layer(temp_input)
temp_out.sum().backward()
print("PrunableLinear sanity check passed!")
print("  output shape:", temp_out.shape)
print("  weight.grad shape:", temp_layer.weight.grad.shape)
print("  gate_scores.grad shape:", temp_layer.gate_scores.grad.shape)

PrunableLinear sanity check passed!
  output shape: torch.Size([2, 4])
  weight.grad shape: torch.Size([4, 8])
  gate_scores.grad shape: torch.Size([4, 8])


## Step 3 – Feed-Forward Network Using `PrunableLinear`

Architecture: `3072 → 1024 → 512 → 256 → 10`  
All linear layers are `PrunableLinear` so every weight has a gate.

**`sparsity_loss()`** returns the **L1 norm** (sum of absolute values) of *all* gate values
across every PrunableLinear layer in the network.  
Because sigmoid outputs are always positive, |gate| = gate, so it is simply the sum.

In [3]:
class SelfPruningNet(nn.Module):
    def __init__(self, dropout_p=0.3):
        super(SelfPruningNet, self).__init__()

        self.flatten_layer = nn.Flatten()

        self.layer1   = PrunableLinear(3072, 1024)
        self.bn1      = nn.BatchNorm1d(1024)
        self.relu1    = nn.ReLU()
        self.drop1    = nn.Dropout(p=dropout_p)

        self.layer2   = PrunableLinear(1024, 512)
        self.bn2      = nn.BatchNorm1d(512)
        self.relu2    = nn.ReLU()
        self.drop2    = nn.Dropout(p=dropout_p)

        self.layer3   = PrunableLinear(512, 256)
        self.bn3      = nn.BatchNorm1d(256)
        self.relu3    = nn.ReLU()
        self.drop3    = nn.Dropout(p=dropout_p)

        self.out_layer = PrunableLinear(256, 10)

    def forward(self, x):
        x = self.flatten_layer(x)

        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.drop1(x)

        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.drop2(x)

        x = self.layer3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.drop3(x)

        x = self.out_layer(x)
        return x

    def get_all_prunable_layers(self):
        list_of_layers = []
        for module in self.modules():
            if isinstance(module, PrunableLinear):
                list_of_layers.append(module)
        return list_of_layers

    def sparsity_loss(self):
        total_l1 = torch.tensor(0.0)

        all_layers = self.get_all_prunable_layers()
        for one_layer in all_layers:
            gate_values = one_layer.get_gates()
            layer_l1    = gate_values.sum()
            total_l1    = total_l1 + layer_l1

        return total_l1

    def sparsity_level(self, threshold=1e-2):
        count_pruned = 0
        count_total  = 0

        with torch.no_grad():
            all_layers = self.get_all_prunable_layers()
            for one_layer in all_layers:
                gate_vals    = one_layer.get_gates()
                for val in gate_vals.flatten():
                    count_total += 1
                    if val.item() < threshold:
                        count_pruned += 1

        if count_total == 0:
            return 0.0
        sparsity_percent = 100.0 * count_pruned / count_total
        return sparsity_percent


temp_net   = SelfPruningNet()
temp_img   = torch.randn(4, 3, 32, 32)
temp_logit = temp_net(temp_img)
print("Network output shape:", temp_logit.shape)
print("Number of PrunableLinear layers:", len(temp_net.get_all_prunable_layers()))
del temp_net, temp_img, temp_logit

Network output shape: torch.Size([4, 10])
Number of PrunableLinear layers: 4


## Step 4 – Load CIFAR-10 Dataset

Standard training augmentation (random crop + horizontal flip) and
per-channel normalisation to zero-mean unit-variance.

In [4]:
def get_cifar10_loaders(batch_size=128):
    channel_mean = (0.4914, 0.4822, 0.4465)
    channel_std  = (0.2470, 0.2435, 0.2616)

    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(channel_mean, channel_std),
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(channel_mean, channel_std),
    ])

    train_dataset = torchvision.datasets.CIFAR10(
        root="./data", train=True,  download=True, transform=train_transform)
    test_dataset  = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=test_transform)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True)
    test_loader  = DataLoader(
        test_dataset,  batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)

    return train_loader, test_loader


train_loader, test_loader = get_cifar10_loaders(batch_size=128)
print("CIFAR-10 loaded successfully")
print("Training batches :", len(train_loader))
print("Test     batches :", len(test_loader))

100%|██████████| 170M/170M [00:03<00:00, 48.7MB/s]


CIFAR-10 loaded successfully
Training batches : 391
Test     batches : 79


## Step 5 – Training and Evaluation Functions

**Total Loss = CrossEntropyLoss + λ × SparsityLoss**

- `CrossEntropyLoss` drives the network to classify correctly.
- `λ × SparsityLoss` penalises the network for having too many active (non-zero) gates.
- A higher λ means stronger pressure to prune → sparser network, possibly lower accuracy.

In [5]:
def train_one_epoch(model, loader, optimizer, criterion, lam_value, device, epoch_num):
    model.train()

    total_loss_sum  = 0.0
    cls_loss_sum    = 0.0
    spar_loss_sum   = 0.0
    correct_count   = 0
    total_samples   = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)

        cls_loss = criterion(logits, labels)

        spar_loss = model.sparsity_loss()

        total_loss = cls_loss + lam_value * spar_loss

        total_loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        total_loss_sum += total_loss.item()
        cls_loss_sum   += cls_loss.item()
        spar_loss_sum  += (lam_value * spar_loss).item()

        _, predicted = logits.max(1)
        total_samples += labels.size(0)
        correct_count += predicted.eq(labels).sum().item()

        if (batch_idx + 1) % 100 == 0:
            running_acc = 100.0 * correct_count / total_samples
            print(f"    [Epoch {epoch_num}  Batch {batch_idx+1}/{len(loader)}]  "
                  f"Loss={total_loss.item():.4f}  "
                  f"cls={cls_loss.item():.4f}  "
                  f"spar={lam_value * spar_loss.item():.4f}  "
                  f"Acc={running_acc:.2f}%")

    num_batches       = len(loader)
    avg_total_loss    = total_loss_sum / num_batches
    avg_cls_loss      = cls_loss_sum   / num_batches
    avg_spar_loss     = spar_loss_sum  / num_batches
    train_accuracy    = 100.0 * correct_count / total_samples

    return avg_total_loss, avg_cls_loss, avg_spar_loss, train_accuracy


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()

    loss_total    = 0.0
    correct_count = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits      = model(images)
        loss_total += criterion(logits, labels).item()

        _, predicted  = logits.max(1)
        total_samples += labels.size(0)
        correct_count += predicted.eq(labels).sum().item()

    avg_loss = loss_total / len(loader)
    accuracy = 100.0 * correct_count / total_samples
    return avg_loss, accuracy

## Step 6 – Run Training for Three λ Values

We compare three λ settings:
- **Low (λ = 0.0001):** very little sparsity pressure → accurate but dense
- **Medium (λ = 0.001):** balanced trade-off
- **High (λ = 0.01):** strong sparsity pressure → very sparse, some accuracy loss

In [ ]:
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

LAMBDA_LOW    = 0.0001
LAMBDA_MEDIUM = 0.001
LAMBDA_HIGH   = 0.01
LAMBDA_LIST   = [LAMBDA_LOW, LAMBDA_MEDIUM, LAMBDA_HIGH]
LAMBDA_LABELS = ["Low",       "Medium",      "High"]

NUM_EPOCHS  = 30
LEARN_RATE  = 1e-3
SAVE_DIR    = "./outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

all_results   = []
all_histories = []


for exp_idx in range(len(LAMBDA_LIST)):
    lam_value  = LAMBDA_LIST[exp_idx]
    lam_label  = LAMBDA_LABELS[exp_idx]
    print(f"  Experiment: λ = {lam_value}  ({lam_label})")

    model     = SelfPruningNet(dropout_p=0.3).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARN_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    history = {"test_acc": [], "sparsity": []}
    best_test_acc = 0.0
    save_path = os.path.join(SAVE_DIR, f"best_lambda_{lam_label.lower()}.pt")

    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start_time = time.time()

        tr_loss, cls_l, spar_l, tr_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, lam_value, device, epoch)

        te_loss, te_acc = evaluate_model(model, test_loader, criterion, device)

        current_sparsity = model.sparsity_level(threshold=1e-2)

        scheduler.step()

        history["test_acc"].append(te_acc)
        history["sparsity"].append(current_sparsity)

        elapsed = time.time() - epoch_start_time
        print(f"  Epoch {epoch:02d}/{NUM_EPOCHS}  "
              f"TrainAcc={tr_acc:.2f}%  "
              f"TestAcc={te_acc:.2f}%  "
              f"Sparsity={current_sparsity:.1f}%  "
              f"({elapsed:.1f}s)")

        if te_acc > best_test_acc:
            best_test_acc = te_acc
            torch.save(model.state_dict(), save_path)

    model.load_state_dict(torch.load(save_path, weights_only=True))
    final_sparsity  = model.sparsity_level(threshold=1e-2)
    _, final_acc    = evaluate_model(model, test_loader, criterion, device)

    print(f"\n  >>> Final Test Accuracy : {final_acc:.2f}%")
    print(f"  >>> Final Sparsity      : {final_sparsity:.2f}%")

    result_dict = {
        "label"    : lam_label,
        "lambda"   : lam_value,
        "accuracy" : final_acc,
        "sparsity" : final_sparsity
    }
    all_results.append(result_dict)
    all_histories.append((lam_label, lam_value, history, model))

Using device: cuda
  Experiment: λ = 0.0001  (Low)
    [Epoch 1  Batch 100/391]  Loss=183.5070  cls=1.7857  spar=181.7213  Acc=27.01%
    [Epoch 1  Batch 200/391]  Loss=177.3113  cls=1.8486  spar=175.4627  Acc=29.89%
    [Epoch 1  Batch 300/391]  Loss=173.1448  cls=1.6676  spar=171.4772  Acc=31.72%
  Epoch 01/30  TrainAcc=33.07%  TestAcc=41.55%  Sparsity=0.0%  (66.0s)
    [Epoch 2  Batch 100/391]  Loss=169.8702  cls=1.8042  spar=168.0660  Acc=36.98%
    [Epoch 2  Batch 200/391]  Loss=169.0217  cls=1.5941  spar=167.4276  Acc=37.97%
    [Epoch 2  Batch 300/391]  Loss=168.6888  cls=1.5606  spar=167.1282  Acc=38.52%
  Epoch 02/30  TrainAcc=38.87%  TestAcc=45.00%  Sparsity=0.0%  (64.7s)
    [Epoch 3  Batch 100/391]  Loss=168.5550  cls=1.6123  spar=166.9427  Acc=41.36%
    [Epoch 3  Batch 200/391]  Loss=168.6595  cls=1.7423  spar=166.9172  Acc=41.07%
    [Epoch 3  Batch 300/391]  Loss=168.4124  cls=1.5063  spar=166.9061  Acc=40.86%
  Epoch 03/30  TrainAcc=41.16%  TestAcc=46.64%  Sparsity=0.0

## Step 7 – Results Summary Table

In [ ]:
print(f"  {'Lambda Label':<14} {'Lambda Value':<15} {'Test Accuracy':<16} {'Sparsity Level'}")
for result in all_results:
    print(f"  {result['label']:<14} {result['lambda']:<15} "
          f"{result['accuracy']:.2f}%{'':>10} {result['sparsity']:.2f}%")

## Step 8 – Training Curves & Gate Distribution Plot



In [ ]:
fig1, (ax_acc, ax_spar) = plt.subplots(1, 2, figsize=(14, 5))
color_list = ["tab:blue", "tab:orange", "tab:green"]

for i in range(len(all_histories)):
    lam_label_i  = all_histories[i][0]
    lam_value_i  = all_histories[i][1]
    history_i    = all_histories[i][2]
    color_i      = color_list[i]
    epoch_list   = list(range(1, len(history_i["test_acc"]) + 1))

    ax_acc.plot(epoch_list, history_i["test_acc"],
                color=color_i, label=f"λ={lam_value_i} ({lam_label_i})")
    ax_spar.plot(epoch_list, history_i["sparsity"],
                 color=color_i, label=f"λ={lam_value_i} ({lam_label_i})")

ax_acc.set_xlabel("Epoch")
ax_acc.set_ylabel("Test Accuracy (%)")
ax_acc.set_title("Test Accuracy vs Epoch")
ax_acc.legend()
ax_acc.grid(alpha=0.3)

ax_spar.set_xlabel("Epoch")
ax_spar.set_ylabel("Sparsity (%)")
ax_spar.set_title("Sparsity Level vs Epoch")
ax_spar.legend()
ax_spar.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curves.png"), dpi=150)
plt.show()
print("Training curves saved.")


best_idx = 0
best_acc_so_far = 0.0
for i in range(len(all_results)):
    if all_results[i]["accuracy"] > best_acc_so_far:
        best_acc_so_far = all_results[i]["accuracy"]
        best_idx = i

best_lam_label = all_histories[best_idx][0]
best_lam_value = all_histories[best_idx][1]
best_model     = all_histories[best_idx][3]
print(f"\nPlotting gate distribution for best model: λ={best_lam_value} ({best_lam_label})")

all_gate_values_list = []
with torch.no_grad():
    for one_layer in best_model.get_all_prunable_layers():
        gate_vals   = one_layer.get_gates().cpu().numpy()
        flat_gates  = gate_vals.flatten().tolist()
        for gv in flat_gates:
            all_gate_values_list.append(gv)

all_gate_values_arr = np.array(all_gate_values_list)

fig2, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle(f"Gate Value Distribution  (Best model: λ={best_lam_value}, {best_lam_label})",
              fontsize=13, fontweight="bold")

ax_full.hist(all_gate_values_arr, bins=100, color="steelblue", edgecolor="none", alpha=0.85)
ax_full.axvline(x=0.01, color="red", linestyle="--", linewidth=1.5, label="Prune threshold (0.01)")
ax_full.set_xlabel("Gate Value")
ax_full.set_ylabel("Count")
ax_full.set_title("All Gate Values")
ax_full.legend()

near_zero_gates = []
for gv in all_gate_values_list:
    if gv < 0.1:
        near_zero_gates.append(gv)
near_zero_arr = np.array(near_zero_gates)

ax_zoom.hist(near_zero_arr, bins=80, color="tomato", edgecolor="none", alpha=0.85)
ax_zoom.axvline(x=0.01, color="darkred", linestyle="--", linewidth=1.5, label="Prune threshold (0.01)")
ax_zoom.set_xlabel("Gate Value")
ax_zoom.set_ylabel("Count")
ax_zoom.set_title("Zoomed: Gates < 0.1")
ax_zoom.legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "gate_distribution.png"), dpi=150)
plt.show()
print("Gate distribution plot saved.")
print(f"Total gates: {len(all_gate_values_list)}")
print(f"Gates below threshold: {len(near_zero_gates)} ({100.0*len(near_zero_gates)/len(all_gate_values_list):.1f}%)")

## Step 10 – Download Output Files (Google Colab only)

In [ ]:
import shutil

try:
    from google.colab import files
    shutil.make_archive("tredence_outputs", "zip", "./outputs")
    files.download("tredence_outputs.zip")
    print("Download started!")
except ImportError:
    print("Not running in Colab — output files are saved locally in ./outputs/")
    print("Files available:")
    for fname in os.listdir(SAVE_DIR):
        print(" ", fname)